# Etapa 7 — LMD_YOLO y YOLOv8n a 100 épocas (Tesla T4)
Misma configuración que la etapa 6 (`configs/comparison.yaml`) salvo las épocas: `configs/stage7_100_epochs.yaml`. Ambos modelos desde cero, seed 42. Ejecute primero solamente **PREFLIGHT + SMOKE**. TEST no se evalúa.

Si Colab se desconecta durante el entrenamiento, vuelva a ejecutar las celdas 1 a 5 y luego la celda **Reanudar** del modelo que se cortó: continúa desde el último `last.pt` guardado en Drive.

In [ ]:
# 1. Comprobar GPU asignada
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Seleccione Runtime > Change runtime type > T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0).upper()

In [ ]:
# 2. Montar Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Localizar y extraer el paquete en almacenamiento rápido temporal
from pathlib import Path
import shutil, zipfile
ZIP_PATH = Path('/content/drive/MyDrive/TP1_LMD/stage7/packages/colab_stage6.zip')
PACKAGE_ROOT = Path('/content/colab_stage6')
RESULTS_ROOT = Path('/content/drive/MyDrive/TP1_LMD/stage7/results')
CONFIG = PACKAGE_ROOT / 'configs' / 'stage7_100_epochs.yaml'
assert ZIP_PATH.is_file(), ZIP_PATH
if PACKAGE_ROOT.exists():
    shutil.rmtree(PACKAGE_ROOT)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')
assert (PACKAGE_ROOT / 'README.md').is_file() and CONFIG.is_file()
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print(PACKAGE_ROOT, RESULTS_ROOT, CONFIG)

In [ ]:
# 4. Verificar todos los hashes del paquete antes de ejecutar código del proyecto
import subprocess, sys
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/manifests.py'), 'verify-package'], check=True)

In [ ]:
# 5. Instalar dependencias generales; NO instala ultralytics ni reemplaza torch CUDA
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE_ROOT/'requirements-colab.txt')], check=True)

In [ ]:
# 6. PREFLIGHT completo: hashes del dataset, modelos, estructura, forward y backward GPU
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'preflight', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 7. SMOKE: 1 época sobre 2% de TRAIN para cada modelo, batch 16
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'smoke', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 8. Revisar el gate. DETENERSE aquí y leer los resultados/VRAM.
import json
preflight = json.loads((RESULTS_ROOT/'preflight.json').read_text())
gate = json.loads((RESULTS_ROOT/'smoke_gate.json').read_text())
print(json.dumps(preflight, indent=2))
print(json.dumps(gate, indent=2))
assert preflight['status'] == gate['status'] == 'PASS'
print('STOP: preflight + smoke completos. No ejecute entrenamiento largo sin revisar estos datos.')

## Gate manual para las 100 épocas
Esta sección no corre por defecto. Solo después de revisar el smoke, cambie conscientemente la variable a `True`.

In [ ]:
RUN_LONG_TRAINING = False  # cambiar manualmente a True tras revisar el smoke
assert RUN_LONG_TRAINING, 'Gate cerrado: no se lanzarán 100 épocas automáticamente.'

In [ ]:
# 9A. LMD_YOLO TRAIN multirrama: 100 épocas desde cero
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'lmd_yolo', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9A-R. Reanudar LMD_YOLO si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'lmd_yolo', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9B. YOLOv8n: 100 épocas desde cero (referencia del paper)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'yolov8n', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 9B-R. Reanudar YOLOv8n si Colab se desconectó (solo en ese caso)
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'train', '--model', 'yolov8n', '--resume', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)

In [ ]:
# 10. Comparación y curvas (requiere los dos modelos); no usa TEST
subprocess.run([sys.executable, str(PACKAGE_ROOT/'scripts/stage6.py'), 'compare', '--results-root', str(RESULTS_ROOT), '--config', str(CONFIG)], check=True)